In [1]:
from gs_lib.gs_tools import *

In [3]:
# illustrate_rotation()
# demo_complete_system()

In [6]:
print("=" * 70)
print("DEMO: BUILDING SIDES AND EXPRESSING MATCHINGS")
print("=" * 70)

# ── 1. CREATE THE TWO SIDES ──────────────────────────
print("\n1. CREATING MEN'S SIDE AND WOMEN'S SIDE")
print("-" * 40)

men = [Man("Adam"), Man("Bob"), Man("Charlie"), Man("David")]
women = [Woman("Eve"), Woman("Fiona"), Woman("Grace"), Woman("Helen")]

print(f"   Men:   {', '.join(m.id for m in men)}")
print(f"   Women: {', '.join(w.id for w in women)}")

# ── 2. EXPRESS PREFERENCES ───────────────────────────
print("\n2. EXPRESSING PREFERENCES")
print("-" * 40)

prefs = PreferenceList({
    # Men's preferences (1st > 2nd > 3rd > 4th)
    men[0]: [women[0], women[1], women[2], women[3]],  # Adam
    men[1]: [women[1], women[0], women[3], women[2]],  # Bob
    men[2]: [women[2], women[3], women[0], women[1]],  # Charlie
    men[3]: [women[3], women[2], women[1], women[0]],  # David
    
    # Women's preferences (1st > 2nd > 3rd > 4th)
    women[0]: [men[3], men[2], men[1], men[0]],        # Eve
    women[1]: [men[2], men[3], men[0], men[1]],        # Fiona
    women[2]: [men[1], men[0], men[3], men[2]],        # Grace
    women[3]: [men[0], men[1], men[2], men[3]],        # Helen
})

print("\n   Men's preferences:")
for m in men:
    ranked = " > ".join(w.id for w in prefs.get_preference(m))
    print(f"     {m.id}: {ranked}")

print("\n   Women's preferences:")
for w in women:
    ranked = " > ".join(m.id for m in prefs.get_preference(w))
    print(f"     {w.id}: {ranked}")

# ── 3. RUN GALE-SHAPLEY ──────────────────────────────
print("\n3. RUNNING GALE-SHAPLEY ALGORITHM")
print("-" * 40)

gs = GaleShapley(prefs)
man_opt, woman_opt = gs.find_both_optimal()

print(f"\n   Man-optimal matching:")
print(f"     {man_opt}")
print("     Satisfaction:")
for m in men:
    w = man_opt.get_partner(m)
    rank = prefs.get_rank(m, w) + 1
    print(f"       {m.id} → {w.id} (his #{rank} choice)")

print(f"\n   Woman-optimal matching:")
print(f"     {woman_opt}")
print("     Satisfaction:")
for w in women:
    m = woman_opt.get_partner(w)
    rank = prefs.get_rank(w, m) + 1
    print(f"       {w.id} → {m.id} (her #{rank} choice)")

# ── 4. VERIFY STABILITY ──────────────────────────────
print("\n4. VERIFYING STABILITY")
print("-" * 40)

verifier = StabilityVerifier(prefs)

for name, matching in [("Man-optimal", man_opt), ("Woman-optimal", woman_opt)]:
    is_stable, reason, blocking = verifier.is_stable(matching)
    status = "✓ STABLE" if is_stable else f"✗ UNSTABLE ({reason})"
    print(f"   {name}: {status}")

# Build an unstable matching to test
all_men_set = set(men)
all_women_set = set(women)
unstable_dict = {men[0]: women[1], men[1]: women[0], men[2]: women[3], men[3]: women[2]}
unstable = Matching.from_dict(unstable_dict, all_men_set, all_women_set)
is_stable, reason, blocking = verifier.is_stable(unstable)
print(f"   Swapped matching: {'✓ STABLE' if is_stable else f'✗ UNSTABLE ({reason})'}")



DEMO: BUILDING SIDES AND EXPRESSING MATCHINGS

1. CREATING MEN'S SIDE AND WOMEN'S SIDE
----------------------------------------
   Men:   Adam, Bob, Charlie, David
   Women: Eve, Fiona, Grace, Helen

2. EXPRESSING PREFERENCES
----------------------------------------

   Men's preferences:
     Adam: Eve > Fiona > Grace > Helen
     Bob: Fiona > Eve > Helen > Grace
     Charlie: Grace > Helen > Eve > Fiona
     David: Helen > Grace > Fiona > Eve

   Women's preferences:
     Eve: David > Charlie > Bob > Adam
     Fiona: Charlie > David > Adam > Bob
     Grace: Bob > Adam > David > Charlie
     Helen: Adam > Bob > Charlie > David

3. RUNNING GALE-SHAPLEY ALGORITHM
----------------------------------------

   Man-optimal matching:
     Matching(pairs=[(Adam-Eve), (Bob-Fiona), (Charlie-Grace), (David-Helen)], unmatched_men=[], unmatched_women=[])
     Satisfaction:
       Adam → Eve (his #1 choice)
       Bob → Fiona (his #1 choice)
       Charlie → Grace (his #1 choice)
       David → Hel

In [ ]:
# ── 5. FIND AND PERFORM ROTATIONS ───────────────────
print("\n5. FINDING AND PERFORMING ROTATIONS")
print("-" * 40)

rotator = RotationOperator(prefs)
rotations = rotator.find_exposed_rotations(man_opt)

print(f"\n   Rotations exposed in man-optimal: {len(rotations)}")

for i, rotation in enumerate(rotations, 1):
    print(f"\n   Rotation {i}:")
    cycle = " → ".join([f"({m.id},{w.id})" for m, w in rotation])
    print(f"     Cycle: {cycle}")
    print(f"     Effect:")
    print(rotator.get_rotation_effect(rotation))
    
    # Perform it
    new_m = rotator.perform_rotation(man_opt, rotation)
    print(f"     Result: {new_m}")
    
    # Check it's stable
    is_stable, _, _ = verifier.is_stable(new_m)
    print(f"     Stable: {'✓' if is_stable else '✗'}")

# Check woman-optimal (should have no rotations)
rotations_w = rotator.find_exposed_rotations(woman_opt)
print(f"\n   Rotations exposed in woman-optimal: {len(rotations_w)}")
print(f"   (Should be 0 — women can't trade up from here)")

# ── 6. BUILD THE LATTICE ─────────────────────────────
print("\n6. BUILDING THE STABLE MATCHING LATTICE")
print("-" * 40)

lattice = StableMatchingLattice(prefs)
lattice.build_lattice()
lattice.print_lattice()

# Compare extremes
lattice.compare_matchings(man_opt, woman_opt)

# ── 7. QUERY MATCHINGS ───────────────────────────────
print("\n7. QUERYING MATCHINGS")
print("-" * 40)

print(f"\n   Man-optimal: {man_opt}")
for m in men:
    w = man_opt.get_partner(m)
    print(f"     {m.id} ↔ {w.id if w else 'unmatched'}")

print(f"\n   Woman-optimal: {woman_opt}")
for w in women:
    m = woman_opt.get_partner(w)
    print(f"     {w.id} ↔ {m.id if m else 'unmatched'}")

# ── 8. UNEQUAL SIDES DEMO ────────────────────────────
print("\n8. UNEQUAL SIDES (3 men, 2 women)")
print("-" * 40)

men_u = [Man("X"), Man("Y"), Man("Z")]
women_u = [Woman("A"), Woman("B")]

prefs_u = PreferenceList({
    men_u[0]: [women_u[0], women_u[1]],
    men_u[1]: [women_u[1], women_u[0]],
    men_u[2]: [women_u[0]],
    women_u[0]: [men_u[1], men_u[0], men_u[2]],
    women_u[1]: [men_u[0], men_u[1]],
})

gs_u = GaleShapley(prefs_u)
m_opt_u = gs_u.find_stable_matching("men")

print(f"\n   Men:   {', '.join(m.id for m in men_u)}")
print(f"   Women: {', '.join(w.id for w in women_u)}")
print(f"\n   Matching: {m_opt_u}")
print(f"   Unmatched men:   {[m.id for m in m_opt_u.unmatched_men]}")
print(f"   Unmatched women: {[w.id for w in m_opt_u.unmatched_women]}")

for m in men_u:
    w = m_opt_u.get_partner(m)
    if w:
        print(f"     {m.id} ↔ {w.id}")
    else:
        print(f"     {m.id} ↔ unmatched")

# Verify
v_u = StabilityVerifier(prefs_u)
stable, reason, _ = v_u.is_stable(m_opt_u)
print(f"\n   Stable: {'✓' if stable else f'✗ ({reason})'}")

print("\n" + "=" * 70)
print("DEMO COMPLETE")
print("=" * 70)